# NSCLC Radiomics Pipeline — NIfTI Test Run

End-to-end pipeline on the **31 pre-converted NIfTI patients** (LUNG1-001 to LUNG1-031):

| Stage | Description |
|-------|-------------|
| **Feature extraction** | PyRadiomics (shape, first-order, GLCM, GLRLM, GLSZM, GLDM) on preprocessed 1mm isotropic CT + GTV masks |
| **Visualisation** | Feature distributions, correlation heatmap |
| **Response labelling** | Binary at 180-day landmark with censoring handling |
| **LASSO selection** | L1-regularised logistic regression (CV-tuned C) to sparsify ~107 features |
| **Classification** | Logistic Regression, Gradient Boosting, Random Forest with ROC, confusion matrices, feature importance |

**Data**: `smb://cmvm.datastore.ed.ac.uk/igmm/AIR-NSCLC/pipeline_output/nifti/NSCLC-Radiomics` (mounted at `/Volumes/AIR-NSCLC/`)

## 1. Setup & Dependencies

In [ ]:
import os, sys, logging, warnings
from pathlib import Path
from typing import Dict, List, Optional, Tuple

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import SimpleITK as sitk

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid", palette="muted")

# Pipeline imports
sys.path.insert(0, str(Path.cwd()))
from radiomics_pipeline.config import (
    PipelineConfig, RadiomicsConfig, HarmonizationConfig,
    EndpointType, FeatureMode,
)
from radiomics_pipeline.preprocessing import ImagePreprocessor, AutomatedQA
from radiomics_pipeline.features.radiomics import RadiomicsExtractor

logging.basicConfig(level=logging.INFO,
                    format="%(asctime)s %(name)s %(levelname)s %(message)s")
logger = logging.getLogger("NSCLC_NIfTI")

print(f"SimpleITK {sitk.Version.VersionString()}")
print(f"NumPy {np.__version__}  |  Pandas {pd.__version__}")

## 2. Paths & Clinical Data

In [ ]:
# --- Paths -----------------------------------------------------------------
NIFTI_ROOT = Path("/Volumes/AIR-NSCLC/pipeline_output/nifti/NSCLC-Radiomics")
CLINICAL_CSV = (
    Path.cwd().parent
    / "01. Data Curation & Preprocessing"
    / "Datasets"
    / "NSCLC-Radiomics"
    / "NSCLC-Radiomics-Lung1.clinical-version3-Oct-2019.csv"
)

assert NIFTI_ROOT.exists(), f"NIfTI root not found: {NIFTI_ROOT}"
assert CLINICAL_CSV.exists(), f"Clinical CSV not found: {CLINICAL_CSV}"

# --- Patient inventory from NIfTI directories ----------------------------
patient_dirs = sorted([d for d in NIFTI_ROOT.iterdir() if d.is_dir()])
patient_ids  = [d.name for d in patient_dirs]
print(f"Found {len(patient_ids)} NIfTI patients: {patient_ids[0]} … {patient_ids[-1]}")

# --- Clinical data --------------------------------------------------------
raw_clinical = pd.read_csv(CLINICAL_CSV)
clinical_df = raw_clinical.rename(columns={
    "PatientID":         "patient_id",
    "Survival.time":     "survival_days",
    "deadstatus.event":  "event",
    "age":               "age",
    "gender":            "gender",
    "Overall.Stage":     "stage",
    "Histology":         "histology",
    "clinical.T.Stage":  "t_stage",
    "Clinical.N.Stage":  "n_stage",
    "Clinical.M.Stage":  "m_stage",
})

# Keep only patients we have NIfTI data for
clinical_df = clinical_df[clinical_df["patient_id"].isin(patient_ids)].copy()
clinical_df["site"] = "LUNG1"          # single-site for now
print(f"Clinical records matched: {len(clinical_df)} / {len(patient_ids)}")
clinical_df.head()

## 3. Pipeline Configuration

In [ ]:
pipeline_config = PipelineConfig(
    target_spacing=(1.0, 1.0, 1.0),
    hu_min=-1000.0,
    hu_max=400.0,
    min_mask_volume_mm3=100.0,
    max_mask_volume_mm3=1_000_000.0,
    test_size=0.20,            # 80/20 split (small dataset)
    random_state=42,
    feature_mode=FeatureMode.RADIOMICS_ONLY,
)

radiomics_config = RadiomicsConfig(
    bin_width=25,
    feature_classes=["shape", "firstorder", "glcm", "glrlm", "glszm", "gldm"],
    normalize=True,
    normalize_scale=100,
    force_2d=False,
)

# Initialise components
preprocessor = ImagePreprocessor(pipeline_config)
qa_checker   = AutomatedQA(pipeline_config)
extractor    = RadiomicsExtractor(config=radiomics_config,
                                  pipeline_config=pipeline_config)

print("Pipeline components ready.")

## 4. Feature Extraction (from NIfTI)

In [ ]:
from tqdm.auto import tqdm

all_features: List[Dict] = []
processing_log: List[Dict] = []

for patient_dir in tqdm(patient_dirs, desc="Extracting features"):
    pid = patient_dir.name
    ct_path   = patient_dir / "ct.nii.gz"
    mask_path = patient_dir / "mask.nii.gz"

    if not ct_path.exists() or not mask_path.exists():
        processing_log.append({"patient_id": pid, "status": "missing_files"})
        continue

    try:
        ct_image   = sitk.ReadImage(str(ct_path))
        mask_image = sitk.ReadImage(str(mask_path))

        # Preprocess (resample, clamp HU)
        ct_proc, mask_proc = preprocessor.preprocess(ct_image, mask_image)

        # Basic QA — check mask has content
        mask_arr = sitk.GetArrayFromImage(mask_proc)
        mask_voxels = int(mask_arr.sum())
        if mask_voxels == 0:
            raise ValueError("Empty mask after preprocessing")

        # Extract radiomic features
        features = extractor.extract(ct_proc, mask_proc, pid)

        # Attach clinical metadata
        clin_row = clinical_df[clinical_df["patient_id"] == pid]
        if not clin_row.empty:
            row = clin_row.iloc[0]
            features["survival_days"] = row.get("survival_days", np.nan)
            features["event"]         = row.get("event", np.nan)
            features["age"]           = row.get("age", np.nan)
            features["stage"]         = row.get("stage", "unknown")
            features["gender"]        = row.get("gender", "unknown")
            features["histology"]     = row.get("histology", "unknown")

        all_features.append(features)
        processing_log.append({"patient_id": pid, "status": "success",
                               "mask_voxels": mask_voxels})
    except Exception as e:
        logger.error(f"{pid}: {e}")
        processing_log.append({"patient_id": pid, "status": "failed",
                               "error": str(e)})

features_df = pd.DataFrame(all_features)
log_df      = pd.DataFrame(processing_log)

n_ok = (log_df["status"] == "success").sum()
print(f"\n✓ Feature extraction complete: {n_ok}/{len(patient_dirs)} patients")
print(f"  Feature matrix shape: {features_df.shape}")

## 5. Visualise Extracted Features

Distribution of selected features, inter-feature correlation, and class balance.

In [ ]:
# ── Identify radiomic feature columns vs metadata ─────────────────────────
feature_cols = [c for c in features_df.columns if c.startswith("original")]
meta_cols    = ["patient_id", "survival_days", "event", "age", "stage",
                "gender", "histology"]

print(f"Radiomic features: {len(feature_cols)}")
for cat in ["shape", "firstorder", "glcm", "glrlm", "glszm", "gldm"]:
    n = sum(1 for c in feature_cols if cat in c.lower())
    print(f"  {cat:>12s}: {n}")

# ── Remove constant / NaN features ───────────────────────────────────────
X_raw = features_df[feature_cols].copy()
valid_mask = (X_raw.std() > 0) & (X_raw.notna().all())
feature_cols = list(X_raw.columns[valid_mask])
print(f"\nValid features after dropping constants / NaNs: {len(feature_cols)}")

In [ ]:
# ── Feature distributions (sample of 12 features) ────────────────────────
sample_feats = feature_cols[::max(1, len(feature_cols) // 12)][:12]
fig, axes = plt.subplots(3, 4, figsize=(16, 9))
for ax, feat in zip(axes.ravel(), sample_feats):
    short_name = feat.split("_", 2)[-1]  # strip "original_shape_" prefix
    features_df[feat].hist(ax=ax, bins=15, edgecolor="white")
    ax.set_title(short_name, fontsize=9)
    ax.tick_params(labelsize=7)
fig.suptitle("Radiomic Feature Distributions (sample)", fontsize=13, y=1.01)
fig.tight_layout()
plt.show()

In [ ]:
# ── Correlation heatmap ───────────────────────────────────────────────────
corr = features_df[feature_cols].corr()

fig, ax = plt.subplots(figsize=(14, 12))
sns.heatmap(corr, cmap="coolwarm", center=0, vmin=-1, vmax=1,
            xticklabels=False, yticklabels=False, ax=ax)
ax.set_title("Radiomic Feature Correlation Matrix", fontsize=13)
fig.tight_layout()
plt.show()

# Report highly correlated pairs
upper = corr.where(np.triu(np.ones(corr.shape, dtype=bool), k=1))
high_corr = [(c1, c2, upper.loc[c1, c2])
             for c1 in upper.index for c2 in upper.columns
             if abs(upper.loc[c1, c2]) > 0.95]
print(f"Feature pairs with |r| > 0.95: {len(high_corr)}")

## 6. Create Response Labels

Define **responder** (PFS ≥ 180 days) vs **non-responder** at a landmark time. Handle censoring by excluding patients censored before the landmark.

In [ ]:
LANDMARK_DAYS = 180  # ~6 months

df = features_df.dropna(subset=["survival_days", "event"]).copy()

# Binary label: 1 = responder (survived ≥ landmark), 0 = non-responder
# Exclude patients censored before landmark (cannot determine label)
censored_before = (df["survival_days"] < LANDMARK_DAYS) & (df["event"] == 0)
df = df[~censored_before].copy()
df["response"] = (df["survival_days"] >= LANDMARK_DAYS).astype(int)

print(f"Patients after censoring filter: {len(df)}")
print(f"Responders:     {(df['response'] == 1).sum()}")
print(f"Non-responders: {(df['response'] == 0).sum()}")

# Visualise class balance & survival distribution
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Class balance — sort by index so 0=Non-resp, 1=Resp
counts = df["response"].value_counts().sort_index()
counts.plot.bar(ax=axes[0], color=["#e74c3c", "#2ecc71"], edgecolor="white")
axes[0].set_xticklabels(["Non-responder", "Responder"], rotation=0)
axes[0].set_ylabel("Count")
axes[0].set_title(f"Response at {LANDMARK_DAYS}-day landmark")

# Survival distribution by response
for label, grp in df.groupby("response"):
    tag = "Responder" if label == 1 else "Non-responder"
    axes[1].hist(grp["survival_days"], bins=15, alpha=0.6, label=tag, edgecolor="white")
axes[1].axvline(LANDMARK_DAYS, color="k", ls="--", label=f"Landmark ({LANDMARK_DAYS}d)")
axes[1].set_xlabel("Survival (days)")
axes[1].set_ylabel("Count")
axes[1].set_title("Survival Distribution by Response")
axes[1].legend()
fig.tight_layout()
plt.show()

## 7. Train / Test Split & Standardisation

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

X = df[feature_cols].values
y = df["response"].values

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s  = scaler.transform(X_test)

print(f"Train: {X_train_s.shape}  (responders {y_train.sum()}/{len(y_train)})")
print(f"Test:  {X_test_s.shape}   (responders {y_test.sum()}/{len(y_test)})")

## 8. LASSO Feature Selection

Use L1-regularised logistic regression with cross-validated regularisation strength to select the most predictive features.

In [ ]:
from sklearn.linear_model import LogisticRegressionCV

# LASSO (L1) with 5-fold CV to pick regularisation strength
# Use a wide range of C values — small dataset needs less regularisation
lasso_cv = LogisticRegressionCV(
    Cs=np.logspace(-4, 4, 30),  # wide range from very strong to very weak reg.
    cv=5,
    penalty="l1",
    solver="saga",
    max_iter=10_000,
    class_weight="balanced",
    scoring="roc_auc",
    random_state=42,
    n_jobs=-1,
)
lasso_cv.fit(X_train_s, y_train)

best_C = lasso_cv.C_[0]
coefs  = lasso_cv.coef_[0]
selected_mask = coefs != 0
selected_features = [feature_cols[i] for i in range(len(feature_cols)) if selected_mask[i]]

print(f"Best C (1/λ): {best_C:.4f}")
print(f"Features selected by LASSO: {len(selected_features)} / {len(feature_cols)}")

# If LASSO still selects nothing, use a manually relaxed C
if len(selected_features) == 0:
    print("\n⚠ LASSO over-regularised — rerunning with relaxed C=1.0")
    from sklearn.linear_model import LogisticRegression as _LR
    _lr = _LR(penalty="l1", solver="saga", C=1.0, max_iter=10_000,
              class_weight="balanced", random_state=42)
    _lr.fit(X_train_s, y_train)
    coefs = _lr.coef_[0]
    selected_mask = coefs != 0
    selected_features = [feature_cols[i] for i in range(len(feature_cols)) if selected_mask[i]]
    best_C = 1.0
    # Update lasso_cv reference for the visualisation cell
    lasso_cv.coef_ = _lr.coef_
    lasso_cv.C_ = np.array([best_C])
    print(f"  Features selected: {len(selected_features)} / {len(feature_cols)}")

In [ ]:
# ── Visualise LASSO results ───────────────────────────────────────────────

# 1) Coefficient path across C values
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Coefficient magnitudes for selected features
sel_coef = pd.DataFrame({
    "feature": selected_features,
    "coefficient": coefs[selected_mask],
}).sort_values("coefficient")

colors = ["#e74c3c" if c < 0 else "#2ecc71" for c in sel_coef["coefficient"]]
short_names = [f.split("_", 2)[-1] for f in sel_coef["feature"]]
axes[0].barh(short_names, sel_coef["coefficient"], color=colors, edgecolor="white")
axes[0].set_xlabel("LASSO Coefficient")
axes[0].set_title(f"LASSO-Selected Features (n={len(selected_features)})")
axes[0].axvline(0, color="k", lw=0.8)
axes[0].tick_params(labelsize=8)

# 2) CV AUC across regularisation strengths
mean_scores = lasso_cv.scores_[1].mean(axis=0)  # class=1
Cs = lasso_cv.Cs_
axes[1].semilogx(Cs, mean_scores, "o-", color="#3498db")
axes[1].axvline(best_C, color="red", ls="--", label=f"Best C = {best_C:.4f}")
axes[1].set_xlabel("C (inverse regularisation)")
axes[1].set_ylabel("Mean CV AUC")
axes[1].set_title("LASSO Regularisation Path")
axes[1].legend()
axes[1].grid(True, alpha=0.3)

fig.tight_layout()
plt.show()

print("\nSelected features:")
for f, c in zip(sel_coef["feature"], sel_coef["coefficient"]):
    print(f"  {f.split('_', 2)[-1]:40s}  coef={c:+.4f}")

## 9. Classifier Training & Model Comparison

Train multiple classifiers on the LASSO-selected feature set and compare performance.

Models:
- **Logistic Regression** (L2) — interpretable baseline
- **Gradient Boosting** — sequential ensemble
- **Random Forest** — bagging ensemble

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import (
    roc_auc_score, roc_curve, f1_score, precision_score,
    recall_score, accuracy_score, classification_report,
    confusion_matrix, ConfusionMatrixDisplay,
)
from sklearn.model_selection import cross_val_score

# ── Reduce to LASSO-selected features ────────────────────────────────────
sel_idx = [i for i, c in enumerate(feature_cols) if c in selected_features]

# If LASSO selected 0 features (over-regularised), fall back to all features
if len(sel_idx) == 0:
    print("⚠ LASSO selected 0 features — using all features instead.")
    sel_idx = list(range(len(feature_cols)))
    selected_features = feature_cols

X_train_sel = X_train_s[:, sel_idx]
X_test_sel  = X_test_s[:, sel_idx]

print(f"Using {len(sel_idx)} features for classification.\n")

# ── Define models ─────────────────────────────────────────────────────────
models = {
    "Logistic (L2)": LogisticRegression(
        C=1.0, penalty="l2", solver="lbfgs", max_iter=5000,
        class_weight="balanced", random_state=42,
    ),
    "Gradient Boosting": GradientBoostingClassifier(
        n_estimators=100, max_depth=3, learning_rate=0.1,
        subsample=0.8, random_state=42,
    ),
    "Random Forest": RandomForestClassifier(
        n_estimators=200, max_depth=5, min_samples_leaf=3,
        class_weight="balanced", random_state=42,
    ),
}

# ── Train, cross-validate, evaluate ──────────────────────────────────────
results = {}
for name, model in models.items():
    # 5-fold CV AUC on training set
    cv_auc = cross_val_score(model, X_train_sel, y_train,
                             cv=5, scoring="roc_auc")
    # Fit on full training set
    model.fit(X_train_sel, y_train)

    # Test set predictions
    y_pred = model.predict(X_test_sel)
    y_prob = (model.predict_proba(X_test_sel)[:, 1]
              if hasattr(model, "predict_proba") else y_pred.astype(float))

    test_auc = roc_auc_score(y_test, y_prob) if len(np.unique(y_test)) > 1 else np.nan

    results[name] = {
        "model": model,
        "cv_auc_mean": cv_auc.mean(),
        "cv_auc_std": cv_auc.std(),
        "test_auc": test_auc,
        "test_f1": f1_score(y_test, y_pred, zero_division=0),
        "test_acc": accuracy_score(y_test, y_pred),
        "test_prec": precision_score(y_test, y_pred, zero_division=0),
        "test_rec": recall_score(y_test, y_pred, zero_division=0),
        "y_pred": y_pred,
        "y_prob": y_prob,
    }
    print(f"{name:20s}  CV-AUC={cv_auc.mean():.3f}±{cv_auc.std():.3f}  "
          f"Test-AUC={test_auc:.3f}  F1={results[name]['test_f1']:.3f}")

## 10. Visualise Model Performance

In [ ]:
# ── ROC curves ────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# (a) ROC curves for each model
ax = axes[0]
for name, res in results.items():
    fpr, tpr, _ = roc_curve(y_test, res["y_prob"])
    ax.plot(fpr, tpr, label=f'{name} (AUC={res["test_auc"]:.3f})')
ax.plot([0, 1], [0, 1], "k--", alpha=0.4, label="Random")
ax.set_xlabel("False Positive Rate")
ax.set_ylabel("True Positive Rate")
ax.set_title(f"ROC Curves — {LANDMARK_DAYS}-day Response")
ax.legend(fontsize=8)
ax.grid(True, alpha=0.3)

# (b) Metric comparison bar chart
ax = axes[1]
comp_df = pd.DataFrame({
    name: {k: v for k, v in res.items() if k.startswith("test_") and k != "test_auc"}
    for name, res in results.items()
}).T
comp_df.columns = [c.replace("test_", "").upper() for c in comp_df.columns]
comp_df.plot.bar(ax=ax, edgecolor="white", rot=15)
ax.set_ylabel("Score")
ax.set_title("Test-Set Metrics Comparison")
ax.set_ylim(0, 1)
ax.legend(fontsize=8)
ax.grid(axis="y", alpha=0.3)

# (c) CV AUC with error bars
ax = axes[2]
model_names = list(results.keys())
cv_means = [results[n]["cv_auc_mean"] for n in model_names]
cv_stds  = [results[n]["cv_auc_std"]  for n in model_names]
bars = ax.barh(model_names, cv_means, xerr=cv_stds,
               color=sns.color_palette("muted", len(model_names)), edgecolor="white")
ax.set_xlabel("AUC-ROC")
ax.set_title("5-Fold Cross-Validation AUC")
ax.set_xlim(0, 1)
ax.grid(axis="x", alpha=0.3)

fig.tight_layout()
plt.show()

In [ ]:
# ── Confusion matrices ────────────────────────────────────────────────────
fig, axes = plt.subplots(1, len(results), figsize=(5 * len(results), 4))
if len(results) == 1:
    axes = [axes]

for ax, (name, res) in zip(axes, results.items()):
    cm = confusion_matrix(y_test, res["y_pred"])
    disp = ConfusionMatrixDisplay(cm, display_labels=["Non-resp", "Responder"])
    disp.plot(ax=ax, cmap="Blues", colorbar=False)
    ax.set_title(name, fontsize=11)

fig.suptitle("Confusion Matrices (Test Set)", fontsize=13, y=1.02)
fig.tight_layout()
plt.show()

## 11. Feature Importance Analysis

In [ ]:
# ── Feature importance from each model ────────────────────────────────────
short_sel = [f.split("_", 2)[-1] for f in selected_features]

fig, axes = plt.subplots(1, 3, figsize=(18, 6))

# Logistic coefficients
lr = results["Logistic (L2)"]["model"]
lr_imp = pd.Series(np.abs(lr.coef_[0]), index=short_sel).sort_values()
lr_imp.tail(15).plot.barh(ax=axes[0], color="#3498db", edgecolor="white")
axes[0].set_title("Logistic Regression |coef|")
axes[0].set_xlabel("Absolute coefficient")

# Gradient Boosting importance
gb_model = results["Gradient Boosting"]["model"]
gb_imp = pd.Series(gb_model.feature_importances_, index=short_sel).sort_values()
gb_imp.tail(15).plot.barh(ax=axes[1], color="#e67e22", edgecolor="white")
axes[1].set_title("Gradient Boosting Feature Importance")
axes[1].set_xlabel("Importance")

# Random Forest importance
rf_model = results["Random Forest"]["model"]
rf_imp = pd.Series(rf_model.feature_importances_, index=short_sel).sort_values()
rf_imp.tail(15).plot.barh(ax=axes[2], color="#2ecc71", edgecolor="white")
axes[2].set_title("Random Forest Feature Importance")
axes[2].set_xlabel("Gini importance")

fig.suptitle("Top-15 Feature Importances by Model", fontsize=13, y=1.01)
fig.tight_layout()
plt.show()

## 12. Summary Results Table

In [ ]:
# ── Summary table ─────────────────────────────────────────────────────────
summary_rows = []
for name, res in results.items():
    summary_rows.append({
        "Model": name,
        "CV AUC (mean±std)": f"{res['cv_auc_mean']:.3f} ± {res['cv_auc_std']:.3f}",
        "Test AUC": f"{res['test_auc']:.3f}",
        "Test F1": f"{res['test_f1']:.3f}",
        "Test Accuracy": f"{res['test_acc']:.3f}",
        "Test Precision": f"{res['test_prec']:.3f}",
        "Test Recall": f"{res['test_rec']:.3f}",
    })
summary_table = pd.DataFrame(summary_rows).set_index("Model")
print("=" * 80)
print(f"NSCLC RADIOMICS PIPELINE — RESULTS SUMMARY")
print(f"  Patients (after censoring filter): {len(df)}")
print(f"  Train / Test: {len(y_train)} / {len(y_test)}")
print(f"  Total radiomic features: {len(feature_cols)}")
print(f"  LASSO-selected features: {len(selected_features)}")
print(f"  Landmark: {LANDMARK_DAYS} days")
print("=" * 80)
summary_table

## 13. Notes — Full Architecture Extension

In the full hybrid pipeline the workflow extends as follows:

1. **Deep feature extraction** — A pretrained 3D encoder (e.g. ResNet-50 or ViT-3D) produces a 512-dim latent vector per patient.  
2. **Feature concatenation** — Radiomic features (~107 dims, post-LASSO ~10–30) are concatenated with deep features (512 dims) giving a ~520–620-dim vector.  
3. **ComBat harmonisation** — If multi-site data is involved, `ComBatHarmonizer.fit_transform()` corrects batch effects on the concatenated vector, fitted on training data only.  
4. **Classification** — The same LASSO → classifier pipeline runs on the harmonised hybrid vector.  

The `HybridFeaturePipeline` and `FeatureIntegrator` classes in the `radiomics_pipeline` package handle steps 1–3.  
This notebook demonstrates the radiomics-only path as a baseline before adding the deep features.